<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Check the test before the result

**[Check the test before the result](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/)** · Experiments and Causal Impact: Did Our Change Work? · Module 3, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** check a test before you read its result: a chi-square test of the split, the same check by device to find where it breaks, and A/A tests that check the analysis itself.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Welch's t-test and the interval, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); p-values, from the [mathematics course](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M03-L02-check-the-test-before-the-result/check-the-test-before-the-result-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "delivery/customers.csv.gz": "c22521f272d36b36fd26234d53d04aac1c8be347a5014ee3e2aeeffcc34a0c19",
    "delivery/truth.json": "6f088f29a112de123bb1da3bac0940a28c5adb706e9f24007686c373c14ab6a4"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Check the split of a clean test

Hana's free-delivery test planned a 50/50 split: a coin flip for each customer. The chi-square test compares the counts that we see with the counts that we expect. It adds (seen − expected)² / expected over the two groups.

The helper `welch` gives the difference B − A, its 95% interval and the p-value, as `xp analyse` does.

In [ ]:
d = pd.read_csv("data/delivery/customers.csv.gz")

def welch(df, col="revenue"):
    """Difference B - A of a column, its 95% interval and p-value (Welch's t-test)."""
    a, b = df.loc[df.variant == "A", col], df.loc[df.variant == "B", col]
    t = stats.ttest_ind(b, a, equal_var=False)
    ci = t.confidence_interval()
    return round(float(b.mean() - a.mean()), 4), round(float(ci.low), 4), round(float(ci.high), 4), float(f"{t.pvalue:.3g}")

def split_check(df):
    counts = df.variant.value_counts().sort_index()
    res = stats.chisquare(counts, [len(df) / 2] * 2)
    print(f"A {counts['A']:,}, B {counts['B']:,} (B {counts['B'] / len(df):.2%}): chi-square {res.statistic:.2f}, p = {res.pvalue:.3g}")

split_check(d)

> **Check.** You should see `A 55,430, B 55,329 (B 49.95%): chi-square 0.09, p = 0.762`. A fair coin often makes a gap this size: no mismatch.

## 2. Plant a redirect bug

Now we break the test on purpose, as Larkfield's redirect did. The next cell removes some B customers from the log: mobile customers who visited only once, with a chance of 30% each. They stand for customers whose page failed to load. A customer with one visit spent 5.93 dollars on average, against 35.79 for the others.

> **Predict.** Will B's mean revenue in the broken log go up or down? Will desktop still be on plan?

In [ ]:
rng = np.random.default_rng(1)
lost = (d.variant == "B") & (d.device == "mobile") & (d.visits == 1) & (rng.random(len(d)) < 0.3)
broken = d[~lost]
print(f"lost: {lost.sum():,} customers")
split_check(broken)
for device, part in broken.groupby("device"):
    print(device, end=": ")
    split_check(part)

> **Check.** You should see `lost: 3,326 customers`, then `B 48.41%` with p far below 0.001. Desktop stays near 50%; mobile is off plan.

## 3. What the bug does to the result

Compare the estimate of the clean log and of the broken log with the simulator's true effect.

In [ ]:
tr = truth("delivery")["revenue"]["effect"]
print("true effect:", tr)
print("clean log: ", welch(d))
print("broken log:", welch(broken))

> **Check.** You should see `clean log:  (1.0266, 0.1583, 1.895, 0.0205)` and `broken log: (2.4009, 1.5068, 3.295, 1.42e-07)`, against a true effect of 0.6979.

The broken log's interval is as narrow as the clean one, and far from the truth. The customers who were lost spent little, so the B group that is left looks better than it is. With a sample ratio mismatch, do not use the result.

## 4. A/A tests: check the analysis itself

In an A/A test, both groups see the same shop. We can make A/A tests from group A alone: split its customers in two at random, and run Welch's t-test. Do it 1,000 times. A healthy analysis finds p < 0.05 in about 5% of them.

In [ ]:
a_only = d.loc[d.variant == "A", "revenue"].to_numpy()
rng = np.random.default_rng(0)
p_values = []
for _ in range(1000):
    half = rng.random(len(a_only)) < 0.5
    p_values.append(stats.ttest_ind(a_only[half], a_only[~half], equal_var=False).pvalue)
p_values = np.array(p_values)
print(f"false positives: {(p_values < 0.05).mean():.1%}")

import matplotlib.pyplot as plt
plt.hist(p_values, bins=20, range=(0, 1), color="#0054D8")
plt.axvline(0.05, color="#B54708", linestyle="--")
plt.xlabel("p-value of one A/A test"); plt.ylabel("A/A tests")
plt.show()

> **Check.** You should see `false positives: 4.9%` and 20 bars of about the same height. In a healthy analysis, every p-value is equally likely when nothing changes.

## 5. Your turn: the chi-square of mobile

> **Your turn.** Compute the chi-square statistic of the mobile customers in `broken` by hand: (seen − expected)² / expected for A, plus the same for B. The expected count is half of the mobile customers. Put it in `chi_mobile`, rounded to 1 decimal. Then compare it with `stats.chisquare`.

In [ ]:
seen = broken[broken.device == "mobile"].variant.value_counts()
expected = seen.sum() / 2
chi_mobile = round(((seen["A"] - expected) ** 2 + (seen["B"] - expected) ** 2) / expected, 1)
print(seen["A"], seen["B"], chi_mobile, stats.chisquare(seen).statistic.round(1))

In [ ]:
expect('chi_mobile', chi_mobile, 199.6, tolerance=0.001)

## Next

You can check a test before you read it. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/) has the diagrams and the knowledge checks that count toward your certificate. Next, we make a healthy test more precise with the customers' spending before the test.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Get more precise with CUPED](https://learning.nextia-ai.com/courses/experiments/m03/get-more-precise-with-cuped/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.